# Obtención y  remuestreo de datos sobre criptomonedas

In [1]:
import requests
import pandas as pd

#Consulta con la API Kraken en este caso datos de Ethereum
endpoint = 'https://api.kraken.com/0/public/Trades'
payLoad = {'pair': 'XETHZUSD'}
response = requests.get(url=endpoint, params=payLoad)
tradeData = response.json()
trades = tradeData['result']['XETHZUSD']
#Ver las 5 primeras operaciones
trades[:5]

[['2338.74000', '3.25593449', 1773772490.388754, 's', 'l', '', 61534507],
 ['2338.74000', '8.37606551', 1773772490.388754, 's', 'l', '', 61534508],
 ['2338.63000', '0.47940150', 1773772490.388754, 's', 'l', '', 61534509],
 ['2338.63000', '0.64327297', 1773772490.388754, 's', 'l', '', 61534510],
 ['2338.63000', '0.85775247', 1773772490.3887541, 's', 'l', '', 61534511]]

In [2]:
#Crear del dataframe,por defecto las 1000 últimas operaciones
tradesDF = pd.DataFrame.from_records(trades,                                    
columns=['Precio','Volumen','Tiempo','Compra/Venta','Mercado/Orden Límite','Misc','DUMMY'])
tradesDF = tradesDF.drop(columns=['DUMMY']) 
tradesDF

,Precio,Volumen,Tiempo,Compra/Venta,Mercado/Orden Límite,Misc
0,2338.74000,3.25593449,1.773772e+09,s,l,
1,2338.74000,8.37606551,1.773772e+09,s,l,
2,2338.63000,0.47940150,1.773772e+09,s,l,
3,2338.63000,0.64327297,1.773772e+09,s,l,
4,2338.63000,0.85775247,1.773772e+09,s,l,
...,...,...,...,...,...,...
995,2331.74000,7.50231503,1.773776e+09,b,l,
996,2331.74000,0.46237384,1.773776e+09,b,l,
997,2331.74000,9.45478063,1.773776e+09,b,l,
998,2331.74000,0.00348744,1.773776e+09,b,l,


In [3]:
#Crear el indice de tiempo en formato fecha y hora
tradesDF['Tiempo'] = pd.to_datetime(tradesDF['Tiempo'], unit='s')
tradesDF.set_index('Tiempo',inplace=True)
tradesDF

,Precio,Volumen,Compra/Venta,Mercado/Orden Límite,Misc
Tiempo,,,,,
2026-03-17 18:34:50.388753891,2338.74000,3.25593449,s,l,
2026-03-17 18:34:50.388753891,2338.74000,8.37606551,s,l,
2026-03-17 18:34:50.388753891,2338.63000,0.47940150,s,l,
2026-03-17 18:34:50.388753891,2338.63000,0.64327297,s,l,
2026-03-17 18:34:50.388754129,2338.63000,0.85775247,s,l,
...,...,...,...,...,...
2026-03-17 19:32:58.670274496,2331.74000,7.50231503,b,l,
2026-03-17 19:32:58.670274496,2331.74000,0.46237384,b,l,
2026-03-17 19:32:58.670274496,2331.74000,9.45478063,b,l,


In [4]:
#consultar la última marca de tiempo
tradeData["result"]["last"]

'1773775978670274466'

In [5]:
pd.to_datetime(int(tradeData["result"]["last"]),unit="ns")

Timestamp('2026-03-17 19:32:58.670274466')

In [6]:
#Remuestreo por minutos
tradesDF.resample('1min')['Precio'].agg(['first'])

,first
Tiempo,
2026-03-17 18:34:00,2338.74000
2026-03-17 18:35:00,2338.00000
2026-03-17 18:36:00,2339.11000
2026-03-17 18:37:00,2340.07000
2026-03-17 18:38:00,2339.24000
2026-03-17 18:39:00,2339.85000
2026-03-17 18:40:00,2340.50000
2026-03-17 18:41:00,2340.29000
2026-03-17 18:42:00,2340.01000


In [8]:
#Obtener dos ohlc: Open ,High,Low y Close
ohlc = tradesDF.resample("1min", label="left")["Precio"].agg(["first","max","min","last"])
ohlc.columns = ["Open","High","Low","Close"]

In [9]:
ohlc

,Open,High,Low,Close
Tiempo,,,,
2026-03-17 18:34:00,2338.74000,2338.74000,2338.00000,2338.00000
2026-03-17 18:35:00,2338.00000,2339.00000,2337.99000,2339.00000
2026-03-17 18:36:00,2339.11000,2340.80000,2339.11000,2339.77000
2026-03-17 18:37:00,2340.07000,2340.09000,2339.34000,2339.46000
2026-03-17 18:38:00,2339.24000,2339.66000,2338.52000,2339.59000
2026-03-17 18:39:00,2339.85000,2342.00000,2339.85000,2340.70000
2026-03-17 18:40:00,2340.50000,2341.16000,2340.01000,2341.14000
2026-03-17 18:41:00,2340.29000,2340.29000,2340.00000,2340.00000
2026-03-17 18:42:00,2340.01000,2340.78000,2340.01000,2340.01000


## Ejemplo con función

In [10]:

import requests
import pandas as pd
import datetime
from datetime import timezone
import time

def getKrakenTradeData(pair, startDate, endDate):
    endpoint = 'https://api.kraken.com/0/public/Trades'
    
    startTime = int(datetime.datetime.strptime(startDate, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp()*1000000000)
    endTime =   int(datetime.datetime.strptime(endDate, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp()*1000000000)
    
    timeLoaded = startTime
    
    result = pd.DataFrame()
    
    while timeLoaded < endTime:
        print(pd.to_datetime(timeLoaded, unit='ns').strftime('%Y-%m-%d %H:%M:%S'))
        payLoad = {'pair': pair,
                   'since': timeLoaded}
        
        response = requests.get(url=endpoint, params=payLoad)
        data = response.json()['result']
        tradesRaw = data[pair]
        timeLoaded = int(data["last"])
        
        tradeData = pd.DataFrame.from_records(tradesRaw,
                             columns=['Price', 'Volume', 'Time', 'BuySell', 'MarketLimit', 'Misc','DUMMY'])
        tradeData['Time'] = pd.to_datetime(tradeData['Time'], unit='s')
        
        result = pd.concat([result, tradeData])
        
        time.sleep(5)
        
    result.set_index("Time", inplace = True)
    result = result.loc[startDate:endDate+' 00:00:00']
    
    return result

### Utilizando la función , en este caso bitcoin con unas fechas especificas

In [32]:
df = getKrakenTradeData(
        pair="XXBTZUSD",
        startDate="2026-03-11",
        endDate="2026-03-12"
     )
df = df.drop(columns=['DUMMY']) 
df.tail(10)

2026-03-11 00:00:00
2026-03-11 00:17:24
2026-03-11 00:40:02
2026-03-11 01:14:48
2026-03-11 01:47:45
2026-03-11 02:23:24
2026-03-11 02:49:04
2026-03-11 03:21:21
2026-03-11 04:00:56
2026-03-11 04:33:50
2026-03-11 05:17:42
2026-03-11 05:46:27
2026-03-11 06:21:11
2026-03-11 07:05:10
2026-03-11 07:50:50
2026-03-11 08:35:01
2026-03-11 09:04:57
2026-03-11 09:40:13
2026-03-11 10:14:57
2026-03-11 11:00:44
2026-03-11 11:39:13
2026-03-11 11:59:51
2026-03-11 12:12:46
2026-03-11 12:30:12
2026-03-11 12:49:39
2026-03-11 13:13:57
2026-03-11 13:20:07
2026-03-11 13:31:12
2026-03-11 13:38:02
2026-03-11 13:45:07
2026-03-11 13:52:57
2026-03-11 14:00:59
2026-03-11 14:07:17
2026-03-11 14:15:42
2026-03-11 14:27:24
2026-03-11 14:38:44
2026-03-11 14:51:38
2026-03-11 15:06:06
2026-03-11 15:17:09
2026-03-11 15:31:10
2026-03-11 15:45:03
2026-03-11 16:01:03
2026-03-11 16:11:55
2026-03-11 16:27:40
2026-03-11 16:44:40
2026-03-11 17:01:07
2026-03-11 17:13:50
2026-03-11 17:23:42
2026-03-11 17:30:49
2026-03-11 17:39:29


,Price,Volume,BuySell,MarketLimit,Misc
Time,,,,,
2026-03-11 23:59:34.932181358,70200.40000,0.00013864,b,l,
2026-03-11 23:59:50.419239759,70200.30000,0.00056000,s,l,
2026-03-11 23:59:55.885238409,70200.40000,0.00021156,b,l,
2026-03-12 00:00:00.093214273,70200.40000,0.00064980,b,l,
2026-03-12 00:00:00.093214273,70200.40000,0.00005192,b,l,
2026-03-12 00:00:00.124227524,70200.40000,0.00013964,b,l,
2026-03-12 00:00:00.148654461,70200.40000,0.00021262,b,l,
2026-03-12 00:00:00.159912825,70200.40000,0.00070519,b,l,
2026-03-12 00:00:00.343577147,70200.40000,0.00035086,b,l,


In [35]:
#Remuestreo en horas
df.resample('1h')['Price'].agg(['first'])

,first
Time,
2026-03-11 00:00:00,69955.40000
2026-03-11 01:00:00,70052.10000
2026-03-11 02:00:00,70084.50000
2026-03-11 03:00:00,69812.00000
2026-03-11 04:00:00,69567.70000
2026-03-11 05:00:00,70133.20000
2026-03-11 06:00:00,69512.20000
2026-03-11 07:00:00,69955.00000
2026-03-11 08:00:00,69647.10000


In [37]:
# Obtener los datos OHLC: Open,High,Low,Close
ohlc = df.resample("1h", label="left")["Price"].agg(["first","max","min","last"])
ohlc.columns = ["Open","High","Low","Close"]
ohlc

,Open,High,Low,Close
Time,,,,
2026-03-11 00:00:00,69955.40000,70165.20000,69782.10000,70052.10000
2026-03-11 01:00:00,70052.10000,70138.60000,69902.10000,70084.40000
2026-03-11 02:00:00,70084.50000,70121.90000,69644.80000,69803.70000
2026-03-11 03:00:00,69812.00000,69887.00000,69517.10000,69567.60000
2026-03-11 04:00:00,69567.70000,70251.80000,69519.80000,70154.50000
2026-03-11 05:00:00,70133.20000,70186.90000,69450.00000,69512.10000
2026-03-11 06:00:00,69512.20000,69985.30000,69512.10000,69955.00000
2026-03-11 07:00:00,69955.00000,70012.40000,69512.40000,69649.20000
2026-03-11 08:00:00,69647.10000,69793.80000,69512.10000,69665.10000
